# micrograd exercises

1. watch the [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) on YouTube
2. come back and complete these exercises to level up :)

## section 1: derivatives

In [3]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [4]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  ag = -3*a**2 - 0.5*a**(-0.5)
  bg = 3*cos(3*b) + 2.5*b**1.5
  cg = c**-2
  return [ag, bg, cg] # todo, return [df/da, df/db, df/dc]



# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553390593273
OK for dim 1: expected 10.25699027111255, yours returns 10.25699027111255
OK for dim 2: expected 0.0625, yours returns 0.0625


In [5]:
# now estimate the gradient numerically without any calculus, using
# the approximation we used in the video.
# you should not call the function df from the last cell

# -----------
numerical_grad = [0, 0, 0] # TODO
# -----------
h = 0.0001
a = 2
b = 3
c = 4
numerical_grad[0] = (f(a+h,b,c) - f(a,b,c))/h
numerical_grad[1] = (f(a,b+h,c) - f(a,b,c))/h
numerical_grad[2] = (f(a,b,c+h) - f(a,b,c))/h

for dim in range(3):
  ok = 'OK' if abs(numerical_grad[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad[dim]}")


WRONG! for dim 0: expected -12.353553390593273, yours returns -12.354148981312818
WRONG! for dim 1: expected 10.25699027111255, yours returns 10.25712962014147
OK for dim 2: expected 0.0625, yours returns 0.06249843753636242


In [6]:
# there is an alternative formula that provides a much better numerical
# approximation to the derivative of a function.
# learn about it here: https://en.wikipedia.org/wiki/Symmetric_derivative
# implement it. confirm that for the same step size h this version gives a
# better approximation.

# -----------
numerical_grad2 = [0, 0, 0] # TODO
# -----------
numerical_grad2[0] = (f(a+h,b,c) - f(a-h,b,c))/(2*h)
numerical_grad2[1] = (f(a,b+h,c) - f(a,b-h,c))/(2*h)
numerical_grad2[2] = (f(a,b,c+h) - f(a,b,c-h))/(2*h)

for dim in range(3):
  ok = 'OK' if abs(numerical_grad2[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {numerical_grad2[dim]}")


OK for dim 0: expected -12.353553390593273, yours returns -12.353553400714645
OK for dim 1: expected 10.25699027111255, yours returns 10.25699031393934
OK for dim 2: expected 0.0625, yours returns 0.06250000003760192


## section 2: support for softmax

In [30]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  # ------

  def exp(self):
    out = Value(exp(self.data), (self,), "exp")
    def _backward():
      self.grad = out.data * out.grad
    out._backward = _backward
    return out

  def __mul__(self, other):
    out = Value((self.data*other.data), (self,other), "*")
    def _backward():
      self.gard = out.grad * other.data
      other.grad = out.grad * self.data
    out._backward = _backward
    return out

  def __neg__(self):
    out = Value(self.data * (-1), (self,), "neg")
    def _backward():
      self.grad = out.grad * (-1)
    out._backward = _backward
    return out

  def __pow__(self, other):
    out = Value(self.data**other.data, (self, other), "**")
    def _backward():
      self.grad = out.grad * other.data * self.data**(other.data-1)
      other.grad = out.grad * self.data**other.data
    out._backward = _backward
    return out

  def __truediv__(self, other):
    out = Value(data=(self.data * other.data**(-1)), _children=(self, other), _op="/")
    def _backward():
      self.data = out.grad * other.data**(-1)
      other.data = out.grad * self.data * (-1)
    out._backward = _backward
    return out

  def log(self):
    out = Value(log(self.data), (self,), "log")
    def _backward():
      self.grad = out.grad / self.data
    out._backward = _backward
    return out



  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

Let's fix the `Value` class by correcting the typo in `__mul__` and the gradient assignments in `__truediv__`.

In [41]:
# Value class starter code, with many functions taken out
from math import exp, log

class Value:

  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"

  def __add__(self, other): # exactly as in the video
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')

    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward

    return out

  # ------
  # re-implement all the other functions needed for the exercises below
  # your code here
  # TODO
  # ------

  def exp(self):
    out = Value(exp(self.data), (self,), "exp")
    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value((self.data*other.data), (self,other), "*")
    def _backward():
      self.grad += out.grad * other.data # Corrected typo: self.gard to self.grad
      other.grad += out.grad * self.data
    out._backward = _backward
    return out

  def __neg__(self):
    out = Value(self.data * (-1), (self,), "neg")
    def _backward():
      self.grad += out.grad * (-1)
    out._backward = _backward
    return out

  def __pow__(self, other):
    # Ensure 'other' is a Value object for consistent gradient propagation
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data**other.data, (self, other), "**")
    def _backward():
      self.grad += out.grad * other.data * self.data**(other.data-1)
      # For 'other', the gradient depends on log(self.data), so we need to add a log method if it doesn't exist yet
      # Or, for simplicity and assuming 'other' is usually a constant in these examples, we might skip its gradient for now
      # If 'other' is a Value, its gradient would be out.grad * self.data**other.data * log(self.data)
      other.grad += out.grad * self.data**other.data * log(self.data) # Added gradient for other in __pow__
    out._backward = _backward
    return out

  def __truediv__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(data=(self.data / other.data), _children=(self, other), _op="/") # Use / directly
    def _backward():
      self.grad += out.grad / other.data # Corrected: self.data to self.grad
      other.grad += out.grad * (-self.data / (other.data**2)) # Corrected: other.data to other.grad
    out._backward = _backward
    return out

  def log(self):
    out = Value(log(self.data), (self,), "log")
    def _backward():
      self.grad += out.grad / self.data
    out._backward = _backward
    return out

  def __radd__(self, other):
      return self + other

  def __rmul__(self, other):
      return self * other

  def __rpow__(self, other):
      return self**other

  def __sub__(self, other):
      return self + (-other)

  def __rsub__(self, other):
      return other + (-self)

  def __neg__(self):
      return self * -1


  def backward(self): # exactly as in video
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node._backward()


In [40]:
# without referencing our code/video __too__ much, make this cell work
# you'll have to implement (in some cases re-implemented) a number of functions
# of the Value object, similar to what we've seen in the video.
# instead of the squared error loss this implements the negative log likelihood
# loss, which is very often used in classification.

# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts, Value(0.0))
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")


2.1755153626167147
WRONG! for dim 0: expected 0.041772570515350445, yours returns 0.0
WRONG! for dim 1: expected 0.8390245074625319, yours returns 0.0
WRONG! for dim 2: expected 0.005653302662216329, yours returns 0.0
WRONG! for dim 3: expected -0.8864503806400986, yours returns -0.0


Now, let's re-run the softmax and loss calculation to check the gradients with the corrected `Value` class.

In [45]:
# this is the softmax function
# https://en.wikipedia.org/wiki/Softmax_function
def softmax(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts, Value(0.0))
  out = [c / denominator for c in counts]
  return out

# this is the negative log likelihood loss function, pervasive in classification
logits = [Value(0.0), Value(3.0), Value(-2.0), Value(1.0)]
probs = softmax(logits)
loss = -probs[3].log() # dim 3 acts as the label for this input example
loss.backward()
print(loss.data)

ans = [0.041772570515350445, 0.8390245074625319, 0.005653302662216329, -0.8864503806400986]
for dim in range(4):
  ok = 'OK' if abs(logits[dim].grad - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {logits[dim].grad}")

2.1755153626167147
OK for dim 0: expected 0.041772570515350445, yours returns 0.041772570515350445
OK for dim 1: expected 0.8390245074625319, yours returns 0.8390245074625319
OK for dim 2: expected 0.005653302662216329, yours returns 0.005653302662216329
OK for dim 3: expected -0.8864503806400986, yours returns -0.8864503806400988


In [47]:
# verify the gradient using the torch library
# torch should give you the exact same gradient
import torch
def softmaxt(logits):
  counts = [logit.exp() for logit in logits]
  denominator = sum(counts)
  out = [c / denominator for c in counts]
  return out

logits = [torch.tensor([0.0], requires_grad=True), torch.tensor([3.0], requires_grad=True), torch.tensor([-2.0], requires_grad=True), torch.tensor([1.0], requires_grad=True)]
probs = softmaxt(logits)
loss = -probs[3].log()
loss.backward()
print(loss.data)


tensor([2.1755])
